In [12]:
import numpy as np
import random
from collections import deque

# --- НАСТРОЙКИ ---
BOARD_SIZE = 7
WIN_LEN = 5  # Нужно собрать 5 в ряд
EMPTY, X, O = 0, 1, 2
N_ACTIONS = BOARD_SIZE * BOARD_SIZE
INPUT_DIM = BOARD_SIZE * BOARD_SIZE * 3

GAMMA = 0.95
LR = 1e-3
BATCH_SIZE = 64
MEMORY_SIZE = 10000
TARGET_UPDATE = 200
EPS_START = 1.0
EPS_END = 0.1
EPS_DECAY = 0.999
EPISODES = 2000
HIDDEN = 128

def check_win(board, player):
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if board[r, c] != player:
                continue
            for dr, dc in [(0,1), (1,0), (1,1), (1,-1)]:
                cnt, rr, cc = 1, r+dr, c+dc
                while 0 <= rr < BOARD_SIZE and 0 <= cc < BOARD_SIZE and board[rr,cc] == player:
                    cnt += 1
                    if cnt >= WIN_LEN:
                        return True
                    rr += dr; cc += dc
    return False

def is_full(board):
    return not np.any(board == EMPTY)

def available_moves(board):
    return list(zip(*np.where(board == EMPTY)))

def encode_board(board, player):
    opp = O if player == X else X
    out = np.zeros((3, BOARD_SIZE, BOARD_SIZE), dtype=np.float32)
    out[0] = (board == EMPTY)
    out[1] = (board == player)
    out[2] = (board == opp)
    return out.reshape(-1)

class MLP:
    def __init__(self):
        rng = np.random.default_rng(42)
        self.W1 = rng.normal(0, np.sqrt(2/INPUT_DIM), (INPUT_DIM, HIDDEN)).astype(np.float32)
        self.b1 = np.zeros(HIDDEN, dtype=np.float32)
        self.W2 = rng.normal(0, np.sqrt(2/HIDDEN), (HIDDEN, N_ACTIONS)).astype(np.float32)
        self.b2 = np.zeros(N_ACTIONS, dtype=np.float32)
        self.params = ['W1', 'b1', 'W2', 'b2']
        self.m = {k: np.zeros_like(getattr(self, k)) for k in self.params}
        self.v = {k: np.zeros_like(getattr(self, k)) for k in self.params}
        self.t = 0

    def forward(self, x):
        z1 = x @ self.W1 + self.b1
        a1 = np.maximum(z1, 0)
        q  = a1 @ self.W2 + self.b2
        self._cache = (x, z1, a1)
        return q

    def backward(self, gq):
        x, z1, a1 = self._cache
        g = {}
        g['W2'] = a1.T @ gq
        g['b2'] = gq.sum(0)
        dz1 = (gq @ self.W2.T) * (z1 > 0)
        g['W1'] = x.T @ dz1
        g['b1'] = dz1.sum(0)
        return g

    def adam_step(self, g, lr=LR):
        self.t += 1
        for k in self.params:
            self.m[k] = 0.9 * self.m[k] + 0.1 * g[k]
            self.v[k] = 0.999 * self.v[k] + 0.001 * g[k]**2
            mh = self.m[k] / (1 - 0.9**self.t)
            vh = self.v[k] / (1 - 0.999**self.t)
            setattr(self, k, getattr(self, k) - lr * mh / (np.sqrt(vh) + 1e-8))

    def copy_from(self, other):
        for k in self.params:
            setattr(self, k, getattr(other, k).copy())

# --- УЛУЧШЕННЫЙ АГЕНТ DQN С ЗАЩИТНЫМ ПРАВИЛОМ ---
class DQNAgent:
    def __init__(self):
        self.online = MLP()
        self.target = MLP()
        self.target.copy_from(self.online)
        self.memory = deque(maxlen=MEMORY_SIZE)
        self.epsilon = EPS_START
        self.steps = 0

    def act(self, board, player, train=True):
        moves = available_moves(board)
        if not moves:
            return None

        opp = O if player == X else X
        for move in moves:
            test_board = board.copy()
            test_board[move] = opp
            if check_win(test_board, opp):
                return move

        if train and random.random() < self.epsilon:
            return random.choice(moves)

        q = self.online.forward(encode_board(board, player))
        mask = np.full(N_ACTIONS, -1e9, dtype=np.float32)
        for (r, c) in moves:
            idx = r * BOARD_SIZE + c
            mask[idx] = q[idx]
        i = int(np.argmax(mask))
        return (i // BOARD_SIZE, i % BOARD_SIZE)

    def remember(self, s, a, r, s2, done, mask):
        a_idx = a[0]*BOARD_SIZE + a[1]
        self.memory.append((s, a_idx, r, s2, done, mask))

    def train_step(self):
        if len(self.memory) < BATCH_SIZE:
            return
        b = random.sample(self.memory, BATCH_SIZE)
        S  = np.stack([x[0] for x in b])
        A  = np.array([x[1] for x in b])
        R  = np.array([x[2] for x in b], np.float32)
        S2 = np.stack([x[3] for x in b])
        D  = np.array([x[4] for x in b], np.float32)
        M  = np.stack([x[5] for x in b])

        q = self.online.forward(S)
        qsa = q[np.arange(BATCH_SIZE), A]
        qn = self.target.forward(S2)
        qn = np.where(M > 0, qn, -1e9)
        tgt = R + GAMMA * (1 - D) * qn.max(1)

        gq = np.zeros_like(q)
        gq[np.arange(BATCH_SIZE), A] = (qsa - tgt) / BATCH_SIZE
        self.online.adam_step(self.online.backward(gq))

        self.steps += 1
        if self.steps % TARGET_UPDATE == 0:
            self.target.copy_from(self.online)

In [13]:
def train_self_play(agent, episodes=EPISODES):
    print("Начало обучения агента через Self-Play...")
    win_x = win_o = draws = 0

    for ep in range(episodes):
        board = np.zeros((BOARD_SIZE, BOARD_SIZE), np.int8)
        cur_player = X
        history = []
        done = False
        winner = None

        while not done:
            s = encode_board(board, cur_player)
            a = agent.act(board, cur_player, train=True)
            if a is None:
                break

            next_board = board.copy()
            next_board[a] = cur_player

            if check_win(next_board, cur_player):
                winner, done = cur_player, True
            elif is_full(next_board):
                done = True

            s2 = encode_board(next_board, cur_player)
            mask_next = np.zeros(N_ACTIONS, dtype=np.float32)
            for (r,c) in available_moves(next_board):
                mask_next[r*BOARD_SIZE + c] = 1.0

            history.append((s, a, cur_player, s2, done, mask_next))
            board = next_board

            if not done:
                cur_player = O if cur_player == X else X

        for s, a, p, s2, d, m in history:
            if winner is not None:
                reward = 1.0 if p == winner else -1.0
            else:
                reward = 0.1
            agent.remember(s, a, reward, s2, d, m)

        if winner == X:
            win_x += 1
        elif winner == O:
            win_o += 1
        else:
            draws += 1

        agent.train_step()
        agent.epsilon = max(EPS_END, agent.epsilon * EPS_DECAY)

        if (ep + 1) % 500 == 0:
            print(f"Эпизод {ep+1:4d} | Epsilon: {agent.epsilon:.3f} | X побед: {win_x} | O побед: {win_o} | Ничьи: {draws}")
            win_x = win_o = draws = 0

In [14]:
def print_board(board):
    symbols = {EMPTY: '.', X: 'X', O: 'O'}
    print("  " + " ".join(str(i) for i in range(BOARD_SIZE)))
    for r in range(BOARD_SIZE):
        print(f"{r} " + " ".join(symbols[board[r, c]] for c in range(BOARD_SIZE)))
    print()

def play_demo(agent):
    print("\n=== ДЕМОНСТРАЦИОННАЯ ПАРТИЯ АГЕНТА ПРОТИВ СЕБЯ ===")
    board = np.zeros((BOARD_SIZE, BOARD_SIZE), np.int8)
    cur_player = X
    print_board(board)

    while True:
        a = agent.act(board, cur_player, train=False)
        if a is None:
            print("Ничья! Нет доступных ходов.")
            break

        board[a] = cur_player
        print(f"Ход {'X' if cur_player == X else 'O'} в клетку {a}:")
        print_board(board)

        if check_win(board, cur_player):
            print(f"Победа игрока {'X' if cur_player == X else 'O'}!")
            break
        if is_full(board):
            print("Ничья! Поле заполнено.")
            break

        cur_player = O if cur_player == X else X

In [15]:
agent = DQNAgent()
train_self_play(agent, episodes=2000)
play_demo(agent)

Начало обучения агента через Self-Play...
Эпизод  500 | Epsilon: 0.606 | X побед: 11 | O побед: 14 | Ничьи: 475
Эпизод 1000 | Epsilon: 0.368 | X побед: 13 | O побед: 10 | Ничьи: 477
Эпизод 1500 | Epsilon: 0.223 | X побед: 14 | O побед: 8 | Ничьи: 478
Эпизод 2000 | Epsilon: 0.135 | X побед: 15 | O побед: 6 | Ничьи: 479

=== ДЕМОНСТРАЦИОННАЯ ПАРТИЯ АГЕНТА ПРОТИВ СЕБЯ ===
  0 1 2 3 4 5 6
0 . . . . . . .
1 . . . . . . .
2 . . . . . . .
3 . . . . . . .
4 . . . . . . .
5 . . . . . . .
6 . . . . . . .

Ход X в клетку (5, 2):
  0 1 2 3 4 5 6
0 . . . . . . .
1 . . . . . . .
2 . . . . . . .
3 . . . . . . .
4 . . . . . . .
5 . . X . . . .
6 . . . . . . .

Ход O в клетку (3, 3):
  0 1 2 3 4 5 6
0 . . . . . . .
1 . . . . . . .
2 . . . . . . .
3 . . . O . . .
4 . . . . . . .
5 . . X . . . .
6 . . . . . . .

Ход X в клетку (0, 6):
  0 1 2 3 4 5 6
0 . . . . . . X
1 . . . . . . .
2 . . . . . . .
3 . . . O . . .
4 . . . . . . .
5 . . X . . . .
6 . . . . . . .

Ход O в клетку (2, 6):
  0 1 2 3 4 5 6
0 . 

In [16]:
def play_vs_random(agent, play_as=X, num_games=100):
    agent_wins = 0
    random_wins = 0
    draws = 0

    for _ in range(num_games):
        board = np.zeros((BOARD_SIZE, BOARD_SIZE), np.int8)
        cur_player = X
        done = False

        while not done:
            moves = available_moves(board)
            if not moves:
                break

            if cur_player == play_as:
                a = agent.act(board, cur_player, train=False)
            else:
                a = random.choice(moves)

            board[a] = cur_player

            if check_win(board, cur_player):
                if cur_player == play_as:
                    agent_wins += 1
                else:
                    random_wins += 1
                done = True
            elif is_full(board):
                done = True

            cur_player = O if cur_player == X else X

        if not check_win(board, X) and not check_win(board, O):
            draws += 1

    return agent_wins, random_wins, draws

print("=== РАСЧЕТ WINRATE ПРОТИВ СЛУЧАЙНОГО ИГРОКА ===")
wins_as_x, losses_as_x, draws_as_x = play_vs_random(agent, play_as=X, num_games=100)
print(f"Игры за X (Первый ход)  | Победы DQN: {wins_as_x}% | Победы Random: {losses_as_x}% | Ничьи: {draws_as_x}%")

wins_as_o, losses_as_o, draws_as_o = play_vs_random(agent, play_as=O, num_games=100)
print(f"Игры за O (Второй ход)  | Победы DQN: {wins_as_o}% | Победы Random: {losses_as_o}% | Ничьи: {draws_as_o}%")

=== РАСЧЕТ WINRATE ПРОТИВ СЛУЧАЙНОГО ИГРОКА ===
Игры за X (Первый ход)  | Победы DQN: 79% | Победы Random: 2% | Ничьи: 19%
Игры за O (Второй ход)  | Победы DQN: 77% | Победы Random: 1% | Ничьи: 22%


In [17]:
def evaluate_blocking_accuracy(agent, num_tests=100):
    successful_blocks = 0

    for _ in range(num_tests):
        board = np.zeros((BOARD_SIZE, BOARD_SIZE), np.int8)
        direction = random.choice([(0,1), (1,0), (1,1), (1,-1)])
        dr, dc = direction

        start_r = random.randint(0, BOARD_SIZE - 1 - abs(dr)*4)
        start_c = random.randint(4 if dc == -1 else 0, BOARD_SIZE - 1 - max(0, dc)*4)

        line_coords = [(start_r + i*dr, start_c + i*dc) for i in range(5)]

        blocking_idx = random.randint(0, 4)
        block_r, block_c = line_coords[blocking_idx]

        for i, (r, c) in enumerate(line_coords):
            if i != blocking_idx:
                board[r, c] = O

        for _ in range(8):
            empty_cells = available_moves(board)
            if (block_r, block_c) in empty_cells:
                empty_cells.remove((block_r, block_c))
            if empty_cells:
                br, bc = random.choice(empty_cells)
                board[br, bc] = random.choice([X, O])

        a = agent.act(board, player=X, train=False)

        if a == (block_r, block_c):
            successful_blocks += 1

    accuracy = (successful_blocks / num_tests) * 100
    print("=== ПРОВЕРКА ТОЧНОСТИ ЗАЩИТЫ ===")
    print(f"Успешно заблокировано угроз: {successful_blocks}/{num_tests} ({accuracy:.2f}%)")

In [18]:
evaluate_blocking_accuracy(agent)

=== ПРОВЕРКА ТОЧНОСТИ ЗАЩИТЫ ===
Успешно заблокировано угроз: 91/100 (91.00%)
